In [ ]:
import pandas
import pycaret
import numpy as np
#from imblearn.over_sampling import SMOTE
#from sklearn.preprocessing import MinMaxScaler
#import smogn

In [ ]:
filename = "Data_2021_08_12_v1.csv"

In [ ]:
csv = pandas.read_csv(filename)
csv = csv.dropna(axis=0).reset_index(drop=True)#drop NaN
csv = csv[csv['Lmt']>0.1].reset_index(drop=True)
csv = csv[csv['Lmr']>0.1].reset_index(drop=True)

array_len = len(csv)

for i in range(0, array_len): 

    csv['I1'][i] = csv['I1'][i].replace('i', 'j')
    csv['I2'][i] = csv['I2'][i].replace('i', 'j')
    csv['I1'][i] = csv['I1'][i].replace(' ', '')
    csv['I2'][i] = csv['I2'][i].replace(' ', '')
    csv['I1'][i] = complex(csv['I1'][i]).real
    csv['I2'][i] = complex(csv['I2'][i]).real
    

csv['I1'] = pandas.to_numeric(csv['I1'], downcast="float")
csv['I2'] = pandas.to_numeric(csv['I2'], downcast="float")

R1 = np.zeros(array_len)
R2 = np.zeros(array_len)

for i in range(0, array_len): 
    R1[i] = csv['copperloss_tx'][i]/csv['I1'][i]/csv['I1'][i]
    R2[i] = csv['copperloss_rx'][i]/csv['I2'][i]/csv['I2'][i]

csv = csv.assign(R1 = R1)
csv = csv.assign(R2 = R2)

# Data post processing for classification

Lmt_th_up = 20
Lmt_th_low = 5
csv['Lmt'] = csv['Lmt'].apply(lambda x:1 if (x<Lmt_th_up) and (x>Lmt_th_low) else 0)
csv['Lmt'] = pandas.to_numeric(csv['Lmt'], downcast="float")

Lmr_th_up = 20
Lmr_th_low = 5
csv['Lmr'] = csv['Lmr'].apply(lambda x:1 if (x<Lmr_th_up) and (x>Lmr_th_low) else 0)
csv['Lmr'] = pandas.to_numeric(csv['Lmr'], downcast="float")

Llt_th_up = 25
Llt_th_low = 15
csv['Llt'] = csv['Llt'].apply(lambda x:1 if (x<Llt_th_up) and (x>Llt_th_low) else 0)
csv['Llt'] = pandas.to_numeric(csv['Llt'], downcast="float")

Llr_th_up = 20
Llr_th_low = 5
csv['Llr'] = csv['Llr'].apply(lambda x:1 if (x<Llr_th_up) and (x>Llr_th_low) else 0)
csv['Llr'] = pandas.to_numeric(csv['Llr'], downcast="float")

R1_th_up = 0.15
R1_th_low = 0.03
csv['R1'] = csv['R1'].apply(lambda x:1 if (x<R1_th_up) and (x>R1_th_low) else 0)
csv['R1'] = pandas.to_numeric(csv['R1'], downcast="float")

R2_th_up = 0.15
R2_th_low = 0.03
csv['R2'] = csv['R2'].apply(lambda x:1 if (x<R2_th_up) and (x>R2_th_low) else 0)
csv['R2'] = pandas.to_numeric(csv['R2'], downcast="float")

# N1,N2,d1,d2,freq,move_tx,move_rx,offset_tx,offset_rx,per,space1,space2,space3,space4,space5,space6,l1,l2,h1,w1,Lmt,Lmr,Llt,Llr,k,Lt,Lr,Lm,Rt,Rr,I1,I2,Zt,Zr,Zm,copperloss_tx,copperloss_rx


csv = csv.drop('Rt',axis=1)
csv = csv.drop('Rr',axis=1)
csv = csv.drop('Zt',axis=1)
csv = csv.drop('Zr',axis=1)
csv = csv.drop('Zm',axis=1)



data = csv


csv = csv.drop('Lmt',axis=1)
csv = csv.drop('Lmr',axis=1)
#csv = csv.drop('Llt',axis=1)
csv = csv.drop('Llr',axis=1)
csv = csv.drop('k',axis=1)
csv = csv.drop('Lt',axis=1)
csv = csv.drop('Lr',axis=1)
csv = csv.drop('Lm',axis=1)
csv = csv.drop('copperloss_tx',axis=1)
csv = csv.drop('copperloss_rx',axis=1)
csv = csv.drop('R1',axis=1)
csv = csv.drop('R2',axis=1)

dataset = csv
dataset = dataset.dropna(axis=0).reset_index(drop=True)



dataset.hist('Llt',bins=50)
#dataset.hist('I2',bins=50)

dataset

In [ ]:
dataset

dataset

In [ ]:
data = dataset.sample(frac=0.9, random_state=786).reset_index(drop=True)
data_unseen = dataset.drop(data.index).reset_index(drop=True)


print('Data for Modeling: ' + str(data.shape))
print('Unseen Data For Predictions: ' + str(data_unseen.shape))

In [ ]:
from pycaret.classification import *
exp_reg101 = setup(data = data, target = 'Llt', session_id=123, silent=True, use_gpu=True) 

In [ ]:
# top3 = [ create_model(name) for name in ["xgboost","lightgbm", "rf"] ]
#top3 = compare_models(n_select = 3) 
lgbm = create_model('lightgbm',num_leaves=60,max_depth=20)

In [ ]:
#tuned = tune_model(lgbm)

In [ ]:
#tuned_top3 = [tune_model(i) for i in top3]

In [ ]:
#bagged_top3 = [ensemble_model(i) for i in tuned_top3]

In [ ]:
#blender = blend_models(estimator_list = top3)

In [ ]:
#best5 = automl(optimize = 'RMSE')

In [ ]:
plot_model(lgbm)

In [ ]:
plot_model(estimator = best5, plot = 'feature_all')

In [ ]:
evaluate_model(lgbm)

In [ ]:
#save_model(lgbm, "ML_model2_v3_2021_08_12_v1_Llt_c")